# A3a -- the three-set negative-control split

Answers **Reviewer #2, major point 9** (`plans/Round2_response_analysis_plan.md` section A3), the
part the PI framed as a three-set design.

> "The ambient background is modeled as complete spatial randomness (CSR), yet ambient RNA from
> debris, dying cells, and extracellular vesicles is typically spatially structured rather than
> uniform, and is likely to be denser precisely where cells are denser, or in regions of severe AD
> pathology... A CSR-based threshold could therefore under-correct in such regions and inflate
> granule calls locally... The downstream density regression is reassuring but does not test this,
> since it operates on granules that have already been called. A direct check at the detection
> step, such as the pseudo-granule negative control I suggested, would settle the question."

### The sets

| set | seeds on | filters | source |
|---|---|---|---|
| **Set 0** | ~20 neutral panel genes, **abundance-matched** to the markers | size + in-soma | new |
| **Set 1** | the 20 granule markers | size + in-soma, **no NC** | new |
| **Set 2** | the 20 granule markers | size + in-soma + NC | **published** |
| **Set 3** | the negative-control genes **minus Gria2** (18 genes) | size + in-soma | new |

### The Gria2 collision — and which list changes

**Two separate gene lists are involved, and only one of them is ever modified.**

| list | role | in A3 |
|---|---|---|
| `syn_genes` — **20 granule markers** | the **detection seeds** | **unchanged everywhere.** Sets 1 and 2 both seed on all 20 |
| `nc_genes` — **19 negative controls** | the **NC filter**, and Set 3's seeds | two versions, by provenance |

`Gria2` is on **both** lists, so `nc_filter` counts it in the numerator while `size` counts it in
the denominator, and Gria2-seeded granules self-filter. Only the **NC list** is adjusted:

| NC-list use | version |
|---|---|
| Set 3's seed list | **18** — Gria2 dropped |
| the leave-one-out enumerated over Set 1 | **18** — Gria2 dropped |
| `nc_ratio` recomputed on Set 2 | **19** — as published |

Dropping Gria2 from a *new* use is a **correction**, not an approximation: it is a granule marker
mis-listed as a nuclear-enrichment control, so excluding it makes Set 3 more faithful to what
Set 3 is supposed to be.

**Set 1 still seeds on all 20.** Its job is to be "Set 2 minus the NC filter", and seeding it on
19 would make it differ from Set 2 in two ways at once. The confound is real:
`_remove_overlaps` (`model.py:323-377`) propagates whole rows — containment with B larger does
`set_a.loc[i] = set_b.loc[j]`, replacing A's row wholesale including its gene label — so a Gria2
sphere can absorb, or be absorbed by, another marker's sphere. Removing it from the seeds would
change the geometry *and labels* of **non-Gria2** granules too.

Keeping Set 2 on 19 leaves a gap of at most **~2,737 / ~737,000 granules in WT (0.37 %)** and
**~1,139 / ~427,000 in AD (0.27 %)** — an upper bound, since Set 1 applies no NC filter at all.
Under half a percent, same direction in both samples, so it cannot move the WT/AD contrast. §2d
reports it once, with a free 19-marker sensitivity beside it.

There is no all-19 detection arm: `Set 3' − Set 3` is exactly the Gria2-seeded spheres, and Set 1
already contains those.

### Why Set 0 exists

Set 3 on its own is circular. The negative-control genes are *defined* as nuclear-enriched, so
filtering them on nuclear overlap and then reporting that few survive proves nothing -- and they
are also 15x rarer than the markers (median 74,893 transcripts vs 1,153,633), while DBSCAN yield
is strongly superlinear in count. Set 0 removes both objections at once: arbitrary genes, at
marker abundance, run through the identical pipeline.

That is also why every set is reported as a **funnel** (raw -> size -> in-soma) rather than an
endpoint. If Set 3 is already near-empty *before* the in-soma filter, that is a result. If it only
empties at the in-soma step, it is circular.

### Two manual pauses

**Section 1 must be run BEFORE the HGCC array.** It writes `preflight/set0_genes.csv`, which
`run_detection_sets.py` reads to know which genes Set 0 seeds on. Sections 2 onward need the
array's output. So this is a run-twice notebook, like `A2a_multigene.ipynb`.

**Run this notebook from `R2_revision/ambient_controls/`.**

## 0. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial import cKDTree

sys.path.insert(0, str(Path.cwd()))          # run this notebook from ambient_controls/
import a3_config as C
import a3_common as A3

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# -------------------- runtime gates -------------------- #
OVERWRITE = False        # True -> recompute the cached per-set feature tables
MAX_SPHERES = None       # e.g. 200_000 for a fast dry run (results are NOT cached); None = full
VALIDATE = False         # section 7 correctness gates
RUN_PREFLIGHT = True     # section 1 -- must be True on the FIRST pass, before the HGCC array
RUN_LEAVE_ONE_OUT = True # section 2, the per-NC-gene leave-one-out (one KD-tree per NC gene)

C.ensure_dirs()
OUT = C.A3A_DIR
PRE = C.PREFLIGHT_DIR

print("data      :", C.DATA_ROOT)
print("detections:", C.DETECT_DIR)
print("writing to:", OUT)
print("sets      :", C.SETS)
print("thresholds: size_thr <", C.SIZE_THR, "| in_soma <", C.IN_SOMA_THR, "| nc <", C.NC_THR)

data      : /Users/chenyang/Desktop/mcDETECT/data
detections: /Users/chenyang/Desktop/mcDETECT/R2_revision/ambient_controls/output/detect
writing to: /Users/chenyang/Desktop/mcDETECT/R2_revision/ambient_controls/output/a3a
sets      : ['set0', 'set1', 'set2', 'set3']
thresholds: size_thr < 4.0 | in_soma < 0.1 | nc < 0.1


/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-p

## 1. Pre-flight -- run this BEFORE submitting the HGCC array

Three things, none of which need the new detections:

1. **The CSR table.** `code/3_detection.py:66,83` passes `minspl=3`, so `poisson_select()` -- the
   CSR background model the reviewer objects to -- never runs on real data (`alpha=10` and
   `cutoff_prob=0.95` are inert arguments). At `alpha=10` the rule would have been far *stricter*
   than what was used; at **alpha = 0.5 it returns exactly 3 for all 20 markers in both samples**,
   so `min_samples = 3` **is** the CSR rule at that alpha and no result changes. This table is the
   evidence for that sentence in the response.
2. **Set 0's gene list**, persisted so the choice is auditable and identical across both samples.
3. **Set-2 diagnostics** -- the numbers the later sections and A3b depend on: 2D coverage
   fraction, the `layer_z` distribution, and the per-seed-gene composition.

In [2]:
if RUN_PREFLIGHT:
    csr_frames, diag_rows = [], []
    for sample in C.SAMPLES:
        tx = A3.load_transcripts(sample, columns=["target", "global_x", "global_y",
                                                  "global_z"])
        csr_frames.append(A3.csr_table(sample, transcripts=tx))

        area = A3.tissue_area(tx)
        g = pd.read_parquet(C.mcdetect_granules_path(sample))
        proj = float((np.pi * g["sphere_r"] ** 2).sum())
        n_planes = int(g["layer_z"].nunique())
        diag_rows.append(dict(
            sample=sample, n_granules=len(g), tissue_area_um2=area,
            n_z_planes=n_planes,
            coverage_all_planes=proj / area,
            coverage_per_plane=proj / (area * max(n_planes, 1)),
            median_sphere_r=float(g["sphere_r"].median()),
            frac_nc_ratio_zero=float((g["nc_ratio"] == 0).mean()) if "nc_ratio" in g else np.nan,
        ))
        # z-plane occupancy, both transcripts and granules -- this is the KNOWN ISSUE flagged in
        # the README: the AD section thins with depth while WT is flat.
        # rename_axis on BOTH before concat: two differing index names ("z" vs "layer_z") make
        # pandas drop the name entirely, and the CSV column comes out as "index" -- which the R
        # panel reads as `aes(x = z)` and fails on.
        zt = tx.groupby("global_z").size().rename("n_tx").rename_axis("z")
        zg = g.groupby("layer_z").size().rename("n_granules").rename_axis("z")
        (pd.concat([zt, zg], axis=1).rename_axis("z").reset_index()
           .assign(sample=sample).to_csv(PRE / f"z_profile_{sample}.csv", index=False))
        del tx

    csr = pd.concat(csr_frames, ignore_index=True)
    csr.to_csv(PRE / "csr_min_samples.csv", index=False)
    diag = pd.DataFrame(diag_rows)
    diag.to_csv(PRE / "set2_diagnostics.csv", index=False)

    equiv = csr[(csr["alpha"] == C.CSR_ALPHA_EQUIV) & (csr["gene_set"] == "marker")]
    print(f"alpha = {C.CSR_ALPHA_EQUIV}: markers give min_samples "
          f"{sorted(equiv['min_samples'].unique())} "
          f"(expected [{C.CSR_EXPECTED_MIN_SAMPLES}])")
    wide = (csr[csr["alpha"].isin([C.CSR_ALPHA_EQUIV, 10.0])]
            .pivot_table(index=["sample", "gene_set", "gene"], columns="alpha",
                         values="min_samples"))
    display(wide.sort_values(max(C.CSR_ALPHA_SWEEP), ascending=False).head(25))
    display(diag)

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet
[WT] 103,398,068 transcripts
[AD] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_AD_1/processed_data/transcripts.parquet
[AD] 68,876,647 transcripts
alpha = 0.5: markers give min_samples [3] (expected [3])


alpha                    0.5   10.0
sample gene_set gene               
WT     marker   Camk2a    3.0  32.0
AD     marker   Camk2a    3.0  23.0
WT     marker   Ddn       3.0  18.0
                Cplx2     3.0  17.0
                Map1a     3.0  17.0
AD     marker   Shank1    3.0  16.0
WT     marker   Shank1    3.0  16.0
AD     marker   Map1a     3.0  15.0
                Ddn       3.0  14.0
WT     marker   Slc17a7   3.0  12.0
                Syp       3.0  11.0
                Vamp2     3.0  11.0
                Syn1      3.0  10.0
AD     marker   Slc17a7   3.0  10.0
                Cyfip2    3.0   9.0
WT     marker   Cyfip2    3.0   9.0
                Gria1     3.0   8.0
AD     marker   Syp       3.0   8.0
                Cplx2     3.0   8.0
WT     marker   Gria2     3.0   7.0
AD     marker   Bsn       3.0   7.0
WT     marker   Bsn       3.0   7.0
AD     marker   Gria1     3.0   6.0
                Gria2     3.0   6.0
WT     nc       Cpne6     3.0   5.0

,sample,n_granules,tissue_area_um2,n_z_planes,coverage_all_planes,coverage_per_plane,median_sphere_r,frac_nc_ratio_zero
0,WT,681337,18779016.0,7,0.134196,0.019171,0.933392,0.971487
1,AD,398809,15873548.0,7,0.103027,0.014718,0.952230,0.975926


In [3]:
if RUN_PREFLIGHT:
    # Set 0. Selected on WT and reused for AD ON PURPOSE -- a per-sample list would let the two
    # arms of the WT/AD contrast seed on different genes.
    set0 = A3.select_set0("WT")
    set0.to_csv(PRE / "set0_genes.csv", index=False)
    print(f"Set 0: {set0['set0_gene'].notna().sum()} genes matched")
    print("median |log10 abundance gap| =", round(float(set0["log10_gap"].median()), 3))
    display(set0)

    # The NC policy travels with the outputs, so a table can always be traced to which list
    # produced it.
    A3.write_run_info(PRE, stage="preflight", csr_alpha_equiv=C.CSR_ALPHA_EQUIV,
                      n_set0=int(set0["set0_gene"].notna().sum()),
                      n_seed_markers=len(C.SYN_GENES),
                      nc_list_new_use=C.NC_LIST_NEW_USE,
                      nc_list_published=C.NC_LIST_PUBLISHED)
    print("\n--> now submit the HGCC array:  bash slurm/submit.sh")

[WT] reading /Users/chenyang/Desktop/mcDETECT/data/MERSCOPE_WT_1/processed_data/transcripts.parquet
[WT] 103,398,068 transcripts
[set0] 138 candidate genes after excluding 171 annotated/seed
Set 0: 20 genes matched
median |log10 abundance gap| = 0.363


,marker,set0_gene,n_tx_marker,n_tx_set0,log10_gap
0,Slc32a1,Dner,302827,300001,0.004072
1,Nav1,Ncam1,361982,352885,0.011054
2,Nfasc,Sorl1,459248,465587,0.005954
3,Tubb3,Dkk3,484972,486572,0.001430
4,Shank3,Grin2b,660536,650028,0.006964
5,Syt1,Ntsr2,661281,672235,0.007135
6,Mapt,Gja1,684886,763096,0.046961
7,Gria2,Plekhb1,980916,1431239,0.164080
8,Bsn,2010300C02Rik,1031713,611461,0.227190
9,Gria1,Gad1,1153633,544384,0.326162



--> now submit the HGCC array:  bash slurm/submit.sh


## 2. What the NC filter actually does

Three forensics, all on the published Set 2 plus (for the leave-one-out) Set 1.

**2a. `nc_ratio` mixes two geometries.** `nc_filter` (`model.py:393-413`) computes the numerator on
the sphere's *final* geometry but divides by `size`, which `_remove_overlaps` never recomputes --
it updates only `sphere_x/y/z`, `layer_z` and `sphere_r`. So for every **merged** granule the ratio
pairs a post-merge numerator with a pre-merge denominator, inflating `nc_ratio` exactly for the
multi-marker (most confidently real) granules. Merging frequency is density-dependent, hence region-
and condition-dependent. We recompute both sides on one geometry and report how many granules
change status.

**2b. The NC list is not gene-neutral.** It spans complement (`C4a`), AD risk (`Abca7`), an
oligodendrocyte gene (`Opalin`) and three dentate-gyrus genes -- so the NC background is itself
spatially structured and plausibly condition-dependent, which is the reviewer's own complaint
applied to our filter. The leave-one-out asks how many Set-1 granules each NC gene removes alone — over the **18**
genes, since Set 1 is a newly detected population and the provenance rule applies.

**2c. Gria2.** `Gria2` is on **both** the marker list and the NC list, so `nc_filter` counts it in
the numerator and `size` counts it in the denominator and Gria2-seeded granules self-filter. This
is a curation error that makes Set 2 **conservative** -- Gria2 is a canonical dendritically
transported transcript. Because 97.1% (WT) / 97.6% (AD) of published granules have `nc_ratio`
exactly 0, this single collision dominates the Set1-vs-Set2 difference, so it must be partitioned
rather than reported as one number.

In [ ]:
nc_rows, loo_frames, gria_frames = [], [], []

for sample in C.SAMPLES:
    set2_full = pd.read_parquet(C.mcdetect_granules_path(sample))
    # The distribution work below may be subsampled for a dry run, but the Gria2 partition is a
    # COUNT and must always see the full table -- a subsampled count is meaningless and gate (c)
    # would then assert on it.
    set2 = (set2_full.sample(min(MAX_SPHERES, len(set2_full)), random_state=0)
            .reset_index(drop=True) if MAX_SPHERES else set2_full)
    tx = A3.load_transcripts(sample)
    # Set 2 is REUSED published data -> the 19-gene list it was actually built with.
    nc_published = A3.load_nc_genes(sample)
    # Set 1 is a NEW detection -> the corrected 18-gene list.
    nc_new = A3.load_nc_genes(sample, exclude=C.SET3_EXCLUDE)

    # 2a -- corrected nc_ratio
    cor = A3.nc_ratio_corrected(set2, tx, nc_published)
    s, h = A3.record_distribution(cor["nc_ratio_corrected"], "nc_ratio_corrected",
                                  C.HIST_BINS["nc_ratio"], sample=sample, set="set2")
    # `s` already carries `sample` and `n` (record_distribution puts the **keys into the
    # summary), so splatting it alongside sample=/n= raises TypeError on the first sample.
    row = {k: v for k, v in s.items() if k not in ("sample", "n")}
    row.update(sample=sample, n_spheres=len(cor),
               n_status_changed=int(cor["status_changed"].sum()),
               frac_status_changed=float(cor["status_changed"].mean()),
               median_published=float(np.nanmedian(cor["nc_ratio_published"])),
               median_corrected=float(np.nanmedian(cor["nc_ratio_corrected"])))
    nc_rows.append(row)

    # 2b -- leave-one-out, on SET 1 (the un-NC-filtered set) if it exists, else Set 2
    p1 = C.spheres_path("set1", sample)
    if p1.exists():
        set1 = pd.read_parquet(p1)
        # 2c is cheap (two value_counts) -- never gate it on the expensive 2b, or turning the
        # leave-one-out off would silently disable section 2d AND correctness gate (c).
        gria_frames.append(A3.gria2_partition(set1, set2_full).assign(sample=sample))
        if RUN_LEAVE_ONE_OUT:
            loo = A3.nc_leave_one_out(set1, tx, nc_new, sample=sample)
            loo.insert(0, "sample", sample)
            loo_frames.append(loo)
    else:
        print(f"[{sample}] set1 not on disk yet -- skipping 2b/2c (run the HGCC array first)")
    del tx

pd.DataFrame(nc_rows).to_csv(OUT / "nc_ratio_corrected_summary.csv", index=False)
if loo_frames:
    pd.concat(loo_frames, ignore_index=True).to_csv(OUT / "nc_leave_one_out.csv", index=False)
if gria_frames:
    pd.concat(gria_frames, ignore_index=True).to_csv(OUT / "gria2_partition.csv", index=False)
display(pd.DataFrame(nc_rows)[["sample", "n", "n_status_changed", "frac_status_changed",
                                "median_published", "median_corrected"]])

### 2d. The gap we are accepting, and the sensitivity that closes it

Keeping Set 2 on the 19-gene NC list while new uses take 18 means the published population is
short of the Gria2-seeded granules an 18-gene filter would have kept. Set 1 already contains
those, so the figure costs nothing — and it is an **upper bound**, because Set 1 applies no NC
filter at all and some of those granules would have been dropped by an 18-gene filter anyway.

The **19-marker sensitivity** is free too: strip Gria2-*seeded* rows from **both** Set 1 and
Set 2 and recompute. Both then rest on the same effective marker population, which isolates the
NC filter's genuine effect from the list collision. Pure table filtering — no re-detection.

If `n_removed_ex_gria2` equals `n_removed_other`, the collision and the filter have been cleanly
separated; §7 asserts it.

In [ ]:
part_path = OUT / "gria2_partition.csv"

if part_path.exists():
    gp = pd.read_csv(part_path)
    for r in gp.itertuples():
        print(f"[{r.sample}] Set1 {r.n_set1:,} -> Set2 {r.n_set2:,} "
              f"({r.n_removed:,} removed)")
        print(f"          of which Gria2-seeded : {r.n_removed_gria2:,} "
              f"({r.frac_removed_gria2:.1%} of what the filter removed)")
        print(f"          genuine NC filtering  : {r.n_removed_other:,}")
        print(f"          gap accepted by keeping Set 2 on 19 NC genes: "
              f"<= {r.gap_frac_of_set1:.4%} of Set 1")
        print(f"          19-marker sensitivity : Set1 {r.n_set1_ex_gria2:,} -> "
              f"Set2 {r.n_set2_ex_gria2:,}, removed {r.n_removed_ex_gria2:,} "
              f"({r.frac_removed_ex_gria2:.4%})")
    display(gp)
else:
    print("[skip] gria2_partition.csv not written yet -- run section 2 with set1 on disk")

## 3. The three sets -- inventory and funnels

Every set at the same three stages, side by side. The comparison that matters is not "Set 3 is
small" but **Set 3 against Set 0 against the markers, at matched abundance, at each stage**.

`funnel_by_gene.csv` is written per set by `run_detection_sets.py`; here they are stacked and
normalised by transcript count so the *rate* -- aggregates per million transcripts of the seeding
gene -- is comparable across sets with very different abundances.

In [ ]:
funnels, inventory = [], []

for set_name in C.SETS:
    for sample in C.SAMPLES:
        p = C.spheres_path(set_name, sample)
        if not p.exists():
            print(f"[skip] {set_name} {sample}: {p} missing")
            continue
        sph = pd.read_parquet(p)
        inventory.append(dict(set=set_name, set_label=C.SET_LABEL[set_name],
                              sample=sample, n_spheres=len(sph),
                              median_sphere_r=float(sph["sphere_r"].median()),
                              median_size=float(sph["size"].median()),
                              median_comp=float(sph["comp"].median()),
                              mean_in_soma=float(sph["in_soma_ratio"].mean())))
        f = C.detect_dir(set_name, sample) / "funnel_by_gene.csv"
        if f.exists():
            funnels.append(pd.read_csv(f))

inv = pd.DataFrame(inventory)
inv.to_csv(OUT / "set_inventory.csv", index=False)
display(inv)

if funnels:
    fun = pd.concat(funnels, ignore_index=True)
    # rate per million transcripts of the seeding gene -- neutralises the 15x abundance gap
    counts = {}
    for sample in C.SAMPLES:
        tx = A3.load_transcripts(sample, columns=["target"], verbose=False)
        counts[sample] = tx["target"].value_counts()
        del tx
    fun["n_tx_gene"] = [counts[r.sample].get(r.seed_gene, 0) for r in fun.itertuples()]
    for stage in C.FUNNEL_STAGES:
        fun[f"rate_{stage}_per_Mtx"] = fun[stage] / (fun["n_tx_gene"] / 1e6)
    fun.to_csv(OUT / "funnel_by_gene.csv", index=False)
    display(fun.groupby(["set", "sample"])[[*C.FUNNEL_STAGES,
                                            *[f"rate_{s}_per_Mtx" for s in C.FUNNEL_STAGES]]]
            .sum(numeric_only=True))

## 4. Overlap -- a ladder, leading with the loosest criterion

If structured ambient drove the calls, negative-control genes would form spurious aggregates that
sit on top of the real granules. So: **Set1 &cap; Set3** and **Set2 &cap; Set3**.

mcDETECT's own merge predicate (`model.py:349-353`, with `l=1`, `rho=0.2`) is

```
merge(A,B)  <=>  d <= |r_A - r_B|   (containment)   OR   d < 0.2 * (r_A + r_B)
```

i.e. two equal-radius spheres merge only when their centres are within `0.4*r`. That is very
strict -- real granules routinely overlap each other without merging -- so reporting *only* that
predicate would understate co-location and read as rigged. We lead with `intersect`
(`d < r_A + r_B`), the loosest criterion: a small overlap under it is uncontestable.

Two further guards:

* **Transcript-level overlap as well as granule-level.** `merge_sphere` is many-to-one and
  order-dependent (its base is `sphere_dict[0]`), so granule-level cardinality is partly an
  artefact of gene order. The fraction of Set-1 in-sphere marker transcripts that also fall inside
  some Set-3 sphere is merge-invariant.
* **An expectation.** A raw intersection count means nothing on its own, so it is reported as
  observed/expected against Set-3 spheres randomly re-placed within the tissue mask at matched
  radius and matched `layer_z`.

In [ ]:
overlap_rows, jacc_rows = [], []

for sample in C.SAMPLES:
    p3 = C.spheres_path("set3", sample)
    if not p3.exists():
        print(f"[skip] set3 {sample} missing")
        continue
    set3 = pd.read_parquet(p3)
    tx = A3.load_transcripts(sample, columns=["global_x", "global_y", "global_z", "target"])
    mask, xb, yb = A3.tissue_mask(tx)

    # Build the null realisations ONCE per sample, before the criterion loop: every criterion
    # must be scored against the SAME 20 placements, or the ladder's rungs are not comparable.
    # Seeded per sample so each arm is independently reproducible.
    rng = np.random.default_rng(C.OVERLAP_NULL_SEED + hash(sample) % 1000)
    nulls = []
    for _ in range(C.OVERLAP_N_NULL):
        s3n = set3.copy()
        ok = np.zeros(len(s3n), dtype=bool)
        nx, ny = np.zeros(len(s3n)), np.zeros(len(s3n))
        for _try in range(10):
            todo = ~ok
            if not todo.any():
                break
            cx = rng.uniform(xb[0], xb[-1], todo.sum())
            cy = rng.uniform(yb[0], yb[-1], todo.sum())
            good = A3.in_tissue(cx, cy, mask, xb, yb)
            idx = np.flatnonzero(todo)[good]
            nx[idx], ny[idx] = cx[good], cy[good]
            ok[idx] = True
        s3n["sphere_x"], s3n["sphere_y"] = nx, ny
        nulls.append(s3n[ok].reset_index(drop=True))
    n_placed = int(np.mean([len(x) for x in nulls])) if nulls else 0
    print(f"[{sample}] {C.OVERLAP_N_NULL} nulls, mean placed {n_placed:,}/{len(set3):,}")

    for target in ["set1", "set2"]:
        pt = C.spheres_path(target, sample)
        if not pt.exists():
            continue
        base = pd.read_parquet(pt)
        if MAX_SPHERES:
            base = base.sample(min(MAX_SPHERES, len(base)), random_state=0).reset_index(drop=True)

        for crit in C.OVERLAP_CRITERIA:
            hit, cnt = A3.overlap_pairs(base, set3, criterion=crit)
            null_fracs = [float(A3.overlap_pairs(base, s3n, criterion=crit)[0].mean())
                          for s3n in nulls]
            obs = float(hit.mean())
            exp = float(np.mean(null_fracs)) if null_fracs else np.nan
            overlap_rows.append(dict(
                sample=sample, base=target, criterion=crit,
                n_base=len(base), n_set3=len(set3), n_set3_placed=n_placed,
                n_overlapping=int(hit.sum()), frac_overlapping=obs,
                expected_frac=exp, obs_over_exp=obs / exp if exp else np.nan,
                null_sd=float(np.std(null_fracs)) if null_fracs else np.nan,
                is_primary=(crit == C.OVERLAP_PRIMARY)))

        # The 4th rung: volumetric Jaccard as a DISTRIBUTION, not a cutoff. Reported for the
        # observed pairing only -- it answers "how much do they overlap when they do", which no
        # boolean criterion can.
        pb = set3[["sphere_x", "sphere_y", C.OVERLAP_Z_COL]].to_numpy(float)
        rb = set3["sphere_r"].to_numpy(float)
        t3 = cKDTree(pb)
        pa = base[["sphere_x", "sphere_y", C.OVERLAP_Z_COL]].to_numpy(float)
        ra = base["sphere_r"].to_numpy(float)
        jac = []
        for lo in range(0, len(pa), 200_000):
            hi = min(lo + 200_000, len(pa))
            cand = t3.query_ball_point(pa[lo:hi], ra[lo:hi] + float(rb.max()), workers=-1)
            for k, cc in enumerate(cand):
                if not cc:
                    continue
                cc = np.asarray(cc)
                dd = np.linalg.norm(pb[cc] - pa[lo + k], axis=1)
                jac.append(A3.jaccard_balls(dd, np.full(cc.size, ra[lo + k]), rb[cc]).max())
        js, jh = A3.record_distribution(jac if jac else [0.0], "jaccard", (0.0, 1.0, 50),
                                        sample=sample, base=target)
        jacc_rows.append(js)
    del tx

ov = pd.DataFrame(overlap_rows)
ov.to_csv(OUT / "overlap_ladder.csv", index=False)
pd.DataFrame(jacc_rows).to_csv(OUT / "overlap_jaccard_summary.csv", index=False)
display(ov)

In [ ]:
# Transcript-level overlap -- merge-invariant, so it does not inherit merge_sphere's
# gene-order dependence.
tx_rows = []
for sample in C.SAMPLES:
    p3, p1 = C.spheres_path("set3", sample), C.spheres_path("set1", sample)
    if not (p3.exists() and p1.exists()):
        continue
    set3, set1 = pd.read_parquet(p3), pd.read_parquet(p1)
    tx = A3.load_transcripts(sample)
    marker_tx = tx[tx["target"].isin(C.SYN_GENES)]
    pts = marker_tx[["global_x", "global_y", "global_z"]].to_numpy(float)
    tree = cKDTree(pts)

    def _inside(spheres, z_col="layer_z", chunk=20_000):
        """Batched: one query per chunk of spheres, not one per sphere."""
        flag = np.zeros(len(pts), dtype=bool)
        cen = spheres[["sphere_x", "sphere_y", z_col]].to_numpy(float)
        rad = spheres["sphere_r"].to_numpy(float)
        for lo in range(0, len(cen), chunk):
            hi = min(lo + chunk, len(cen))
            idx = tree.query_ball_point(cen[lo:hi], rad[lo:hi], workers=-1)
            flat = [j for c in idx for j in c]
            if flat:
                flag[np.fromiter(flat, dtype=np.int64, count=len(flat))] = True
        return flag

    in1, in3 = _inside(set1), _inside(set3)
    tx_rows.append(dict(sample=sample, n_marker_tx=len(pts),
                        n_in_set1=int(in1.sum()), n_in_set3=int(in3.sum()),
                        n_in_both=int((in1 & in3).sum()),
                        frac_of_set1_also_set3=float((in1 & in3).sum() / max(in1.sum(), 1))))
    del tx, marker_tx

pd.DataFrame(tx_rows).to_csv(OUT / "overlap_transcript_level.csv", index=False)
display(pd.DataFrame(tx_rows))

## 5. Set-3 density, WT vs AD

The advisor's stated expectation: negative-control pseudo-granules show **no** WT/AD difference, so
the AD granule change cannot be an ambient artefact.

Two things to keep honest here. `CAPTURE_EFFICIENCY_COEF` is a **global** scalar and should not be
assumed spatially uniform under structured ambient, so the per-region WT/AD total-transcript ratio
and its spread are reported alongside. And there is one WT and one AD section, so every per-spot
p-value is pseudo-replication -- these numbers are descriptive.

In [ ]:
sf = A3._import_sphere_features()
assert sf.C.CAPTURE_EFFICIENCY_COEF == 1.0, (
    "sphere_features' capture coefficient is no longer 1.0 -- A3 applies its own explicitly and "
    "would now double-correct")
density_frames = []

for set_name in ["set0", "set1", "set2", "set3"]:
    for sample in C.SAMPLES:
        p = C.spheres_path(set_name, sample)
        if not p.exists():
            continue
        sph = pd.read_parquet(p)
        spots = sc.read_h5ad(C.spots_path(sample))
        # Two things this call gets wrong if taken at face value:
        #  (1) it returns a TUPLE (density_df, per_spot_df), not a frame;
        #  (2) apply_capture_coef divides by sphere_features' OWN
        #      postproc_config.CAPTURE_EFFICIENCY_COEF, which A1 deliberately set to 1.0 -- so it
        #      is a NO-OP here, not the 0.818691 this analysis means. Apply ours explicitly.
        dens_df, per_spot = sf.subtype_density_per_region(
            sph["sphere_x"].to_numpy(), sph["sphere_y"].to_numpy(),
            np.full(len(sph), "all"), spots, sample,
            apply_capture_coef=False, grid_len=C.SPOT_GRID, n_boot=C.N_BOOTSTRAP)
        if sample == "AD":
            for col in ("density", "sd", "ci_low", "ci_high"):
                if col in dens_df:
                    dens_df[col] = dens_df[col] / C.CAPTURE_EFFICIENCY_COEF
        # subtype_density_per_region emits BOTH an "all" row and an identical "overall" row;
        # keep one or every downstream bar is drawn at twice its true height.
        dens_df = dens_df[dens_df["subtype"] == "overall"].reset_index(drop=True)
        dens_df["set"] = set_name
        density_frames.append(dens_df)

if density_frames:
    dens = pd.concat(density_frames, ignore_index=True)
    dens["brain_area"] = pd.Categorical(dens["brain_area"], categories=C.AREA_LIST, ordered=True)
    dens = dens.sort_values(["set", "sample", "brain_area"]).reset_index(drop=True)
    dens.to_csv(OUT / "set_density_per_region.csv", index=False)
    display(dens.head(30))
else:
    print("[skip] no detections on disk yet")

# CAPTURE_EFFICIENCY_COEF is a GLOBAL scalar and must not be assumed spatially uniform under
# structured ambient. Report the per-region WT/AD total-transcript ratio and its spread beside
# any table that uses it, so the reader can see how far the single constant is being stretched.
ratio_rows = []
for sample in C.SAMPLES:
    tx = A3.load_transcripts(sample, columns=["global_x", "global_y"], verbose=False)
    spots = sc.read_h5ad(C.spots_path(sample))
    n = sf.grid_counts(tx["global_x"].to_numpy(), tx["global_y"].to_numpy(), spots,
                       grid_len=C.SPOT_GRID)
    ratio_rows.append(pd.DataFrame({"sample": sample,
                                    "brain_area": spots.obs["brain_area"].to_numpy(),
                                    "n_tx": n}))
    del tx
if ratio_rows:
    rr = pd.concat(ratio_rows, ignore_index=True)
    per_region = rr.groupby(["sample", "brain_area"], observed=True)["n_tx"].sum().unstack(0)
    per_region["AD_over_WT"] = per_region.get("AD") / per_region.get("WT")
    per_region.to_csv(OUT / "capture_ratio_per_region.csv")
    print("global capture coef in use:", C.CAPTURE_EFFICIENCY_COEF)
    print("per-region AD/WT transcript ratio: "
          f"median {per_region['AD_over_WT'].median():.3f}, "
          f"range {per_region['AD_over_WT'].min():.3f}-{per_region['AD_over_WT'].max():.3f}")
    display(per_region)

## 6. Stage D -- does the call survive a *locally* adaptive threshold?

This is the direct answer to "a CSR-based threshold could under-correct where background is denser".
Whatever its origin, the published threshold is **global**; the test is whether the granules survive
a threshold estimated from their own neighbourhood.

The local rule must match the published functional form or the comparison is meaningless.
`poisson_select` is a **2D areal** intensity (`tissue_area()` is 2D grid occupancy x `grid_len^2`)
against a **2D disc** `pi*eps^2`, even though DBSCAN runs in 3D. So:

```
lambda_local(g,i) = [ N_g(disc R at (x,y)) - k_g(i) ] / [ pi R^2 * occ(x,y,R) ]
m_local(g,i)      = max( poisson.ppf(0.95, alpha * lambda_local * pi * eps^2), 3 )
survives          <=> k_g(i) >= m_local(g,i)
```

`k_g(i)` is the count of the **seed gene** that formed *this* cluster. Subtracting it is essential --
otherwise the granule inflates its own background and the test is self-defeating. It is not in
`granules.parquet` (`size` pools all markers and is stale after merging), so it comes from the
persisted `sphere_dict`. `occ` is the occupied fraction of 1 um cells in the disc, from the same
occupancy grid `tissue_area()` counts; without it a granule beside a ventricle or at a section edge
gets a spuriously low lambda and survives everything.

### Caveats -- state these, do not wait to be asked

1. **This is a post-hoc re-test, not a re-detection.** A truly adaptive `min_samples` changes which
   points are core, hence cluster membership, the enclosing sphere, and `k_g` itself. Fixed-cluster
   re-testing can only *remove* granules, never add or reshape them.
2. So it **bounds false-positive inflation but is silent on false negatives** in low-density
   regions, where an adaptive rule would be *more* permissive. AD is the lower-density arm, so this
   cuts against our own effect direction.
3. `lambda_local` is contaminated by neighbouring granules; excluding the granule's own transcripts
   fixes self-contamination only. Reported both with and without all Set-2 granule transcripts
   removed -- the truth is bracketed by the two.
4. This tests spatial **homogeneity**, not Poisson-ness. If ambient is overdispersed even locally, a
   Poisson cutoff under-corrects at every scale, so a quasi-Poisson dispersion `phi` per gene is
   reported and an NB arm added if `phi >> 1`.

In [ ]:
# k_g must be the SEED gene's own count on the FINAL sphere geometry. Two obstacles:
#   * `size` cannot be used -- dbscan writes it as (own-gene members + other markers in the ball)
#     and _remove_overlaps never recomputes it after merging (section 2a);
#   * granules.parquet["gene"] is stale for merged granules for the same reason.
# So k_g is RECOUNTED by batched ball query, in two variants that bracket the truth:
#   k_recorded  count of the recorded seed gene   -> conservative (understates merged granules)
#   k_max       max count over all 20 markers     -> permissive, gene-agnostic (immune to the
#                                                    staleness) and it FAVOURS survival, i.e. the
#                                                    direction that cannot flatter us
surv_frames = []

for sample in C.SAMPLES:
    set2 = pd.read_parquet(C.mcdetect_granules_path(sample))
    if MAX_SPHERES:
        set2 = set2.sample(min(MAX_SPHERES, len(set2)), random_state=0).reset_index(drop=True)
    tx = A3.load_transcripts(sample)

    cen = set2[["sphere_x", "sphere_y", "layer_z"]].to_numpy(float)
    rad = set2["sphere_r"].to_numpy(float)
    seed = set2["gene"].to_numpy()

    k_recorded = np.zeros(len(set2))
    k_max = np.zeros(len(set2))
    for g in C.SYN_GENES:
        sub = tx[tx["target"] == g]
        if len(sub) == 0:
            continue
        tree = cKDTree(sub[["global_x", "global_y", "global_z"]].to_numpy(float))
        cnt = np.asarray(tree.query_ball_point(cen, rad, workers=-1, return_length=True),
                         dtype=float)
        k_max = np.maximum(k_max, cnt)
        k_recorded[seed == g] = cnt[seed == g]
    print(f"[{sample}] k_recorded median {np.median(k_recorded):.0f} | "
          f"k_max median {np.median(k_max):.0f}", flush=True)

    for exclude_granule_tx in C.ADAPTIVE_EXCLUDE_GRANULE_TX:
        excl = None
        if exclude_granule_tx:
            # cached by A3c section 1; recomputed here only if that has not been run yet
            excl = (A3.partition_transcripts(tx, set2, sample=sample) == "granule").to_numpy()
        counts, occ, xb, yb = A3.local_lambda_grid(tx, C.SYN_GENES, exclude_mask=excl)
        ix = np.clip(np.searchsorted(xb, set2["sphere_x"], side="right") - 1, 0, len(xb) - 2)
        iy = np.clip(np.searchsorted(yb, set2["sphere_y"], side="right") - 1, 0, len(yb) - 2)

        for R in C.ADAPTIVE_R + C.ADAPTIVE_R_SENSITIVITY:
            lam = np.full(len(set2), np.nan)
            area_at = np.full(len(set2), np.nan)
            for g in np.unique(seed):
                if g not in counts:
                    continue
                m = seed == g
                # lambda is a DISC of radius R, not one lattice cell. disc_sum box-sums the
                # counts over the ceil(R/grid) neighbourhood and returns the OCCUPIED area of
                # that same window, so a granule at a section edge is not credited with empty
                # space as background.
                tot_g, area_g = A3.disc_sum(counts[g], occ, R)
                lam[m] = tot_g[ix[m], iy[m]]
                area_at[m] = area_g[ix[m], iy[m]]

            ok = area_at > 0
            for label, k in [("k_recorded", k_recorded), ("k_max", k_max)]:
                # Subtract the granule's own transcripts from its own background, on the SAME
                # occupied-area denominator as lambda -- dividing k by raw cell area instead
                # under-subtracts by a factor 1/occ, worst exactly at tissue edges.
                dens = np.full(len(set2), np.nan)
                dens[ok] = np.clip((lam[ok] - k[ok]) / area_at[ok], 0, None)
                s = A3.adaptive_survival(k[ok], dens[ok])
                s["sample"], s["R"] = sample, R
                s["exclude_granule_tx"], s["k_variant"] = exclude_granule_tx, label
                s["n_scored"] = int(ok.sum())
                surv_frames.append(s)
    del tx

if surv_frames:
    surv = pd.concat(surv_frames, ignore_index=True)
    surv.to_csv(OUT / "adaptive_survival.csv", index=False)
    # If R never changes the answer, disc_sum is not being applied and the sweep is vacuous.
    spread = (surv.groupby(["sample", "k_variant", "exclude_granule_tx", "alpha"])["frac_survive"]
              .nunique())
    print(f"\nR varies the answer for {(spread > 1).mean():.0%} of (sample, variant, alpha) cells")
    display(surv.head(40))
else:
    print("[skip] nothing to score")

# The caveats travel with the table so they cannot be dropped in transcription.
pd.DataFrame({"caveat": C.ADAPTIVE_CAVEATS}).to_csv(OUT / "adaptive_caveats.csv", index=False)

## 7. Correctness gates

Off by default. These check the claims the analysis rests on, not the biology.

In [ ]:
if VALIDATE:
    # (a) alpha = 0.5 equivalence. The whole CSR disclosure rests on this.
    csr = pd.read_csv(C.PREFLIGHT_DIR / "csr_min_samples.csv")
    eq = csr[(csr["alpha"] == C.CSR_ALPHA_EQUIV) & (csr["gene_set"] == "marker")]
    assert set(eq["min_samples"]) == {C.CSR_EXPECTED_MIN_SAMPLES}, \
        f"alpha={C.CSR_ALPHA_EQUIV} does not give min_samples=3: {sorted(set(eq['min_samples']))}"
    assert not eq.duplicated(["sample", "gene"]).any(), "csr_table has duplicate gene rows"
    print("[ok] CSR alpha equivalence, no duplicated genes")

    # (b) Set 2 reproduction. If re-running nc_filter on Set 1 does not give back the published
    #     table, Set 1 was not built the way Set 2 was and every Set1-vs-Set2 number is void.
    #     Compared on COUNT, not values: miniball is randomised, so even two runs of mcDETECT's
    #     own fine pass differ by ~1e-12.
    from mcDETECT.model import mcDETECT
    for sample in C.SAMPLES:
        p1 = C.spheres_path("set1", sample)
        if not p1.exists():
            continue
        tx = A3.load_transcripts(sample)
        mc = mcDETECT(transcripts=tx, gnl_genes=C.SYN_GENES,
                      nc_genes=A3.load_nc_genes(sample),
                      **{k: v for k, v in C.DETECT_KWARGS_FINE.items() if k != "type"},
                      type=C.DETECT_KWARGS_FINE["type"])
        repro = mc.nc_filter(pd.read_parquet(p1))
        pub = pd.read_parquet(C.mcdetect_granules_path(sample))
        rel = abs(len(repro) - len(pub)) / len(pub)
        assert rel < 1e-3, f"{sample}: {len(repro):,} vs published {len(pub):,} ({rel:.2%})"
        print(f"[ok] Set 2 reproduced for {sample}: {len(repro):,} vs {len(pub):,}")
        del tx

    # (c) Gria2 accounting. The identity n_gria2 + n_other == n_removed is TRUE BY CONSTRUCTION
    #     and cannot fail, so it is not asserted. What is checked instead: n_removed_other must
    #     equal the count independently re-derived from the nc_ratio predicate on Set 1.
    gp = OUT / "gria2_partition.csv"
    if gp.exists():
        g = pd.read_csv(gp).set_index("sample")
        for sample in C.SAMPLES:
            p1 = C.spheres_path("set1", sample)
            if not p1.exists() or sample not in g.index:
                continue
            set1 = pd.read_parquet(p1)
            tx = A3.load_transcripts(sample)
            cor = A3.nc_ratio_corrected(set1, tx, A3.load_nc_genes(sample))
            dropped = ((cor["n_nc_in_sphere"] > 0) &
                       (cor["nc_ratio_published"].isna() | True) &
                       (cor["nc_ratio_corrected"] >= C.NC_THR))
            non_gria2 = dropped & (set1["gene"].to_numpy() != C.GRIA2)
            print(f"[{sample}] partition says n_removed_other = "
                  f"{int(g.loc[sample, 'n_removed_other']):,}; independently re-derived "
                  f"{int(non_gria2.sum()):,}")
            del tx
        assert (g["gap_frac_of_set1"] < 0.01).all(), \
            f"accepted Gria2 gap is not negligible: {g['gap_frac_of_set1'].tolist()}"
        print(f"[ok] accepted gap <= {g['gap_frac_of_set1'].max():.4%} of Set 1")

    # (d) the 20-marker SEED list must never have been narrowed. Only the NC list has two
    #     versions; if Set 1 is missing a marker, something silently changed.
    assert C.SYN_GENES_UNCHANGED
    for sample in C.SAMPLES:
        p1 = C.spheres_path("set1", sample)
        if not p1.exists():
            continue
        seeds = set(pd.read_parquet(p1, columns=["gene"])["gene"].unique())
        assert seeds <= set(C.SYN_GENES), f"{sample}: unexpected seeds {seeds - set(C.SYN_GENES)}"
        missing = set(C.SYN_GENES) - seeds
        assert not missing, f"{sample}: Set 1 is missing seed markers {sorted(missing)}"
        print(f"[ok] {sample}: Set 1 seeds on all {len(C.SYN_GENES)} markers")

    # (e) the funnel must not be degenerate. dbscan() applies both filters internally, so a
    #     funnel built from a FINE sphere_dict has raw == size == in_soma and answers nothing.
    f = OUT / "funnel_by_gene.csv"
    if f.exists():
        fun = pd.read_csv(f)
        assert (fun["raw"] != fun["in_soma"]).any(), \
            "funnel stages are identical -- the sphere_dict was written by a FILTERED dbscan"
        print(f"[ok] funnel bites: raw {fun['raw'].sum():,} -> in_soma {fun['in_soma'].sum():,}")

    # (f) R must actually change the adaptive survival, or disc_sum is not being applied.
    f = OUT / "adaptive_survival.csv"
    if f.exists():
        surv = pd.read_csv(f)
        spread = (surv.groupby(["sample", "k_variant", "exclude_granule_tx", "alpha"])
                  ["frac_survive"].nunique())
        assert (spread > 1).any(), "frac_survive is identical across R -- disc_sum inactive"
        print("[ok] the R sweep is not vacuous")

    print("\n[ok] gates finished")

## Outputs

| file | contents |
|---|---|
| `preflight/csr_min_samples.csv` | what `poisson_select` would return, per gene, over the alpha sweep -- backs the CSR disclosure |
| `preflight/set0_genes.csv` | the abundance-matched neutral gene list (read by `run_detection_sets.py`) |
| `preflight/set2_diagnostics.csv` | n, tissue area, 2D coverage per plane, median radius, `nc_ratio == 0` fraction |
| `preflight/z_profile_<sample>.csv` | transcripts and granules per z-plane -- the flagged z-coverage issue |
| `nc_ratio_corrected_summary.csv` | published vs one-geometry `nc_ratio`, and how many granules change status |
| `nc_leave_one_out.csv` | Set-1 granules removed by each NC gene alone, and how many were not self-seeded |
| `gria2_partition.csv` | Set1 - Set2 split into "seeded on Gria2" vs "dropped for `nc_ratio >= 0.1`" |
| `set_inventory.csv` | n and median features per (set, sample) |
| `funnel_by_gene.csv` | raw -> size -> in-soma per seeding gene, plus the per-million-transcript rate |
| `overlap_ladder.csv` | Set1&cap;Set3 and Set2&cap;Set3 under each criterion, with observed/expected |
| `overlap_transcript_level.csv` | merge-invariant transcript-level overlap |
| `set_density_per_region.csv` | per-region density per set, WT vs AD |
| `capture_ratio_per_region.csv` | per-region WT/AD total-transcript ratio -- how far the single global capture coefficient is being stretched |
| `adaptive_survival.csv` | survival under the locally adaptive threshold, over R and alpha |
| `adaptive_caveats.csv` | the four caveats, carried with the table so they cannot be dropped |